### Import neccessary modules

In [1]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer
from sklearn.model_selection import train_test_split
import time
from tqdm import tqdm
import math

### Loading dataset and Data Preprocessing

In [2]:
EmailDataset = pd.read_csv('CombinedDataSetWithEmailMaximum512Tokens.csv')
phishingEmails = (EmailDataset['label'] == 1).sum()
normalEmails = (EmailDataset['label'] == 0).sum()
print(f"Email Dataset has total {len(EmailDataset)} emails.\nTotal phishing emails: {phishingEmails}\nTotal normal emails: {normalEmails}")

Email Dataset has total 74460 emails.
Total phishing emails: 39419
Total normal emails: 35041


### Spliting Email Dataset into Train/Validation/Test Splits

In [3]:
# We do 70/10/20 -> 70% of the dataset is training data, 10% is validation data, and 20% is testing data

# Obtaining 20% of the dataset as testing data
print("Spliting 20% of Email Dataset to Testing Data...")
trainingAndValidationData, testingData = train_test_split(EmailDataset, test_size=0.2, random_state=42, stratify=EmailDataset['label'])

# Obtaining 70% of the dataset as training data and 10% of the dataset as validation data
print("Spliting 70% of Email Dataset to Training Data and 10% into Validation Data...")
trainingData, validationData = train_test_split(trainingAndValidationData, test_size=0.125, random_state=42, stratify=trainingAndValidationData['label'])

# Resting indices after spliting
#CRUICAL STEP: when we randomly split the dataset, the sub dataset indices are out of order, this will be a problem for PyTorch to load the data later, so we need to reset the splitted data indices to be clean order 0, 1, 2, 3 again
print("Reseting Testing, Training, and Validation Data indices...\n")
trainingData = trainingData.reset_index(drop=True)
validationData = validationData.reset_index(drop=True)
testingData = testingData.reset_index(drop=True)

# Outputing the information of the Training, Testing, and Validation data
print(f"Training Dataset has total {len(trainingData)} emails.\nTotal phishing emails: {(trainingData['label'] == 1).sum()}\nTotal normal emails: {(trainingData['label'] == 0).sum()}\n")
print(f"Validation Dataset has total {len(validationData)} emails.\nTotal phishing emails: {(validationData['label'] == 1).sum()}\nTotal normal emails: {(validationData['label'] == 0).sum()}\n")
print(f"Testing Dataset has total {len(testingData)} emails.\nTotal phishing emails: {(testingData['label'] == 1).sum()}\nTotal normal emails: {(testingData['label'] == 0).sum()}\n")

Spliting 20% of Email Dataset to Testing Data...
Spliting 70% of Email Dataset to Training Data and 10% into Validation Data...
Reseting Testing, Training, and Validation Data indices...

Training Dataset has total 52122 emails.
Total phishing emails: 27593
Total normal emails: 24529

Validation Dataset has total 7446 emails.
Total phishing emails: 3942
Total normal emails: 3504

Testing Dataset has total 14892 emails.
Total phishing emails: 7884
Total normal emails: 7008



### Loading Open-Source Pre-Trained BERT Tokenizer

In [4]:
print("Loading Pre-Trained BERT tokenizer model...")
tokenizer = BertTokenizer.from_pretrained('bert-base-cased') 
print("BERT Tokenizer Loaded Successfully!")
print(f"Vocabulary size: {tokenizer.vocab_size:,} tokens") # Should have 28,996 tokens for 28,996 unique words with cased-sensitve enough to tokenize the email text
print(f"Model max length: {tokenizer.model_max_length}") #Maximum length of 512 words in a text paragraph to be tokenized each iteration

firstEmail = trainingData["email_body_content"][0]
print("\nFirst Email Content")
print()
print(firstEmail)

print("\n" + "=" * 60)

# Tokenize the email
encoded = tokenizer(
    firstEmail,
    add_special_tokens=True,      # Adds [CLS] at start, [SEP] at end
    # max_length=512,                # Maximum sequence length
    padding='max_length',          # Pad shorter sequences to 512
    # truncation=True,               # Cut longer sequences to 512
    return_tensors='pt'            # Return PyTorch tensors
)

print("\nTokenization Results:")
print(f"Input IDs shape: {encoded['input_ids'].shape}")
print(f"Attention Mask shape: {encoded['attention_mask'].shape}")

print(f"\nFirst email token IDs:")
print(encoded['input_ids'][0].tolist())

# print(f"\nFirst email attention mask values:") -> Masking is only for encoder transformer
# print(encoded['attention_mask'][0].tolist()) -> Masking is only for encoder transformer

# Decode back to text to verify
decoded_text = tokenizer.decode(encoded['input_ids'][0], skip_special_tokens=True)
print(f"\nDecoded text:")
print(decoded_text)


Loading Pre-Trained BERT tokenizer model...
BERT Tokenizer Loaded Successfully!
Vocabulary size: 28,996 tokens
Model max length: 512

First Email Content

Male Enhancement


Read hundreds of testimonials from satisfied men and their girlfriends on this site. After years of scientific research, a herbal breakthrough is finally achieved - the product rated as #1 by healthcare professionals, with gains of up to 1-3 inches guaranteed, and girth increases from 5% - 30%.

"I couldn't believe that my Paul gained an incredible 3 inches in just 2 short months. Now that he's longer and also thicker, making love is so much more pleasurable, and I come a lot more easily" Lindsay, California, USA

http://waitfour.com/





Tokenization Results:
Input IDs shape: torch.Size([1, 512])
Attention Mask shape: torch.Size([1, 512])

First email token IDs:
[101, 10882, 13832, 3822, 2093, 1880, 15152, 5229, 1104, 2774, 12013, 21364, 1116, 1121, 8723, 1441, 1105, 1147, 6124, 1116, 1113, 1142, 1751, 119, 1258,

### Recurrent Neural Network

In [5]:
import torch.nn as nn
import torch.optim as optim

In [6]:
# ==========================================
# 1. Custom Dataset Class
# ==========================================
class PhishingEmailDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=100):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        # Returns the number of rows in the dataframe
        return len(self.data)
    

    def __getitem__(self, index):
        # 1. Retrieve the email text and label from the row at the given index
        email_text = str(self.data.loc[index, 'email_body_content'])
        label = self.data.loc[index, 'label']

        encoding = self.tokenizer(
        email_text,               # Text of the email at the current index
        max_length = 50,        # Maximum amount of tokens that can be passed to the tokenizer (512). 100 tokens tends to equal around 75 words. 
        padding = 'max_length',   # If an email is shorter than 512 tokens, it will be padded with 0s out to 512
        truncation = True,        # Any emails longer than 512 tokens will be truncated
        add_special_tokens=True,  # Adds [CLS] at start, [SEP] at end
        return_tensors='pt')      # Return PyTorch tensors


        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(), # Stops model from reading the padding
            'label': torch.tensor(label, dtype=torch.float)
        }
    



In [7]:
# ==========================================
# 2. RNN Model Architecture
# ==========================================
class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, n_layers, dropout):
        super(RNNClassifier, self).__init__()

        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim, padding_idx=0) # Takes the integer ID of a work and replaces it with a vector of decimal numbers
        self.rnn = nn.RNN(input_size=embedding_dim, hidden_size=hidden_dim, num_layers=n_layers, batch_first=True)
        self.dropout = nn.Dropout(p=dropout) # Zeroes out some neruons, helping prevent overfitting (which happens when it is memorizing all the data). P is the probability of zeroing out a neuron
        self.linear = nn.Linear(in_features=hidden_dim, out_features=output_dim) # Takes final memory state of RNN and outpts a number representing probability that the email is phishing

# Forward pass

    def forward(self, input_ids, attention_mask):

        embeddings = self.embedding(input_ids)

        # Pass embeddings through the RNN layer
        rnn_output, hidden = self.rnn(embeddings) # Hidden is the memory state at the end of the sequence. rnn_output includes all the words of the email

        # Extract the output from the last time step

        final_feature = hidden[-1, :, :] # Hidden has the shape [Layers, Batch, Dim]. This line grabs the last layer, all batches, and all dimensions.
        final_feature_post_dropout = self.dropout(final_feature) # Run the dropout function to possibly zero out a neuron

        # Pass through the linear layer to get predictions
        
        final_output = self.linear(final_feature_post_dropout) # Returns a shape of [Batch_Size, 1]

        return final_output.squeeze(1) # squeeze removes extra dimension at the end (turns it into [Batch_Size])



In [ ]:
# ==========================================
# 3. Setup and Hyperparameters
# ==========================================

BATCH_SIZE = 8
HIDDEN_DIM = 256
EMBEDDING_DIM = 128
OUTPUT_DIM = 1 # Binary classification (0 or 1)
N_LAYERS = 2
DROPOUT = 0.4
LEARNING_RATE = 0.0001
EPOCHS = 10

# Total batches of emails that will be used for training the model
TOTAL_BATCHES_OF_TRAINING_EMAILS = math.ceil(len(trainingData) / BATCH_SIZE)

# Instantiate Datasets and DataLoaders
train_dataset = PhishingEmailDataset(trainingData, tokenizer)
val_dataset = PhishingEmailDataset(validationData, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

# Instantiate the Model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = RNNClassifier(
    vocab_size=tokenizer.vocab_size,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    output_dim=OUTPUT_DIM,
    n_layers=N_LAYERS,
    dropout=DROPOUT
)
model = model.to(device)

# Define Loss Function and Optimizer
# Hint: For binary classification with a single output dimension, specific loss functions work best.
criterion = nn.BCEWithLogitsLoss() # Binary Cross Entropy with Logits. Combines sigmoid activation and loss calculation
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [ ]:
# ==========================================
# 4. Training Loop
# ==========================================

# Prints the device being used by Pytorch (either CPU or cuda, cuda == GPU)
print(f'TRAINING START. Using Device: {device}')

for epoch in range(EPOCHS):
    model.train() # Set model to training mode
    total_loss = 0

    print(f"Starting Epoch {epoch+1}...")

    # Progress bar. 70,800 training emails with a batch size of 16 means 4,425 batches per epoch
    with tqdm(total=len(train_loader), desc="Training") as pbar:

    # Training sequence with each batch. 
        for batch in train_loader:
            # 1. Move data to device (GPU/CPU)
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            # 2. Zero gradients (clear them) after each batch
            optimizer.zero_grad() # Call zero_grad on optimizer

            # 3. Forward pass (pass the data through the model to get its predictions)
            predictions = model(input_ids, attention_mask)

            # 4. Calculate Loss (compare the model's predictions to ground truth)
            loss = criterion(predictions, labels)

            # 5. Backward pass (aka Backpropagation. Calculates the gradient of the loss w.r.t. each parameter). This reveals which weights contributed to the error
            loss.backward()

            # 6. Optimizer step (update the weights based on gradients calculated)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) # clip gradient so that model does not update weights more than 1.0 at a time
            optimizer.step()

            total_loss += loss.item()

            pbar.update(1) # Updates the progress bar each time loop completes


    # Calculation of the validation accuracy for this epoch
    model.eval() # Set model to evaluation mode
    with torch.no_grad():
        correct_predictions = 0
        total_predictions = 0

        with tqdm(total=len(val_loader), desc="Evaluating") as pbar:
            for batch in val_loader:

                # Move data to device (GPU/CPU)
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['label'].to(device)

                # Forward pass (pass the data through the model to get its predictions)
                predictions = model(input_ids, attention_mask) # Outputs numbers like [2.1, -1.4, -6.67, etc]. Positive means phishing, negative means normal

                predicted_classes = torch.round(torch.sigmoid(predictions)) # Turns predictions from floats into 0s and 1s

                # Compare predictions to labels. In PyTorch, you can check Tensor A == Tensor B to get a list of trues and falses
                score = (predicted_classes == labels) # Outputs a list like [True, True, False, etc]

                # Adds 1 for each correct prediction to correct prediction counter and adds 1 for each prediction to total prediction counter
                for result in score: 
                    if result == True:
                        correct_predictions +=1
                    total_predictions += 1

                pbar.update(1) # Updates the progress bar each time loop completes

        accuracy = (correct_predictions/total_predictions) * 100 # Calculate accuracy and convert to percentage

    print(f"Epoch {epoch+1} complete. Accuracy: {accuracy:.3f}%\n")

TRAINING START. Using Device: cuda
Starting Epoch 1...


Evaluating: 100%|██████████| 931/931 [00:17<00:00, 52.78it/s]


Epoch 1 complete. Accuracy: 90.787%

Starting Epoch 2...


Evaluating: 100%|██████████| 931/931 [00:17<00:00, 53.90it/s]


Epoch 2 complete. Accuracy: 93.231%

Starting Epoch 3...


...

Epoch 8 complete. Accuracy: 96.199%

Starting Epoch 9...


Evaluating: 100%|██████████| 931/931 [00:17<00:00, 54.28it/s]


Epoch 9 complete. Accuracy: 96.723%

Starting Epoch 10...


Evaluating: 100%|██████████| 931/931 [00:16<00:00, 55.12it/s]

Epoch 10 complete. Accuracy: 97.113%



USE OF AI: 

Used Gemini 2.5 Pro to create code skeleton, debug, explain functions and syntax. Many of the comments are based off its function explainations. My workflow generally consisted of working on each section one at a time, asking Gemini what functions I needed for each section, what arguments they took, what my options were for each function, and what they do. After I filled everything in a section out, I would paste my code to Gemini for debugging. After the program was functional, I would edit hyperparameters and sometimes ask Gemini for ideas to improve the performance. 

PREVIOUS RECORDS: 

5EP, emails 512 tokens or less
TRAINING START.
Starting Epoch 1...
Training: 100%|██████████| 3258/3258 [02:14<00:00, 24.28it/s]
Evaluating: 100%|██████████| 466/466 [00:17<00:00, 26.79it/s]
Epoch 1 complete. Accuracy: 0.8638195004029009

Starting Epoch 2...
Training: 100%|██████████| 3258/3258 [02:16<00:00, 23.84it/s]
Evaluating: 100%|██████████| 466/466 [00:17<00:00, 26.07it/s]
Epoch 2 complete. Accuracy: 0.887187751813054

Starting Epoch 3...
Training: 100%|██████████| 3258/3258 [02:14<00:00, 24.24it/s]
Evaluating: 100%|██████████| 466/466 [00:16<00:00, 28.37it/s]
Epoch 3 complete. Accuracy: 0.9128391082460381

Starting Epoch 4...
Training: 100%|██████████| 3258/3258 [02:16<00:00, 23.79it/s]
Evaluating: 100%|██████████| 466/466 [00:17<00:00, 27.31it/s]
Epoch 4 complete. Accuracy: 0.9235831318828901

Starting Epoch 5...
Training: 100%|██████████| 3258/3258 [02:19<00:00, 23.33it/s]
Evaluating: 100%|██████████| 466/466 [00:17<00:00, 27.16it/s]
Epoch 5 complete. Accuracy: 0.9265377383830244
92.6% acc

5EP, unspecificed token length (took 45 mins)

TRAINING START. Using Device: cuda
Starting Epoch 1...
Training: 100%|██████████| 4425/4425 [08:11<00:00,  9.00it/s]  
Evaluating: 100%|██████████| 633/633 [00:48<00:00, 12.96it/s]
Epoch 1 complete. Accuracy: 87.720%

Starting Epoch 2...
Training: 100%|██████████| 4425/4425 [08:08<00:00,  9.06it/s]  
Evaluating: 100%|██████████| 633/633 [00:51<00:00, 12.29it/s]
Epoch 2 complete. Accuracy: 90.231%

Starting Epoch 3...
Training: 100%|██████████| 4425/4425 [08:11<00:00,  9.01it/s]  
Evaluating: 100%|██████████| 633/633 [00:51<00:00, 12.24it/s]
Epoch 3 complete. Accuracy: 90.825%

Starting Epoch 4...
Training: 100%|██████████| 4425/4425 [08:13<00:00,  8.97it/s]  
Evaluating: 100%|██████████| 633/633 [00:51<00:00, 12.24it/s]
Epoch 4 complete. Accuracy: 93.514%

Starting Epoch 5...
Training: 100%|██████████| 4425/4425 [08:21<00:00,  8.82it/s]  
Evaluating: 100%|██████████| 633/633 [00:49<00:00, 12.68it/s]
Epoch 5 complete. Accuracy: 93.860%

75 tokens, 5 EPCH

TRAINING START. Using Device: cuda
Starting Epoch 1...
Training: 100%|██████████| 3258/3258 [02:13<00:00, 24.35it/s]
Evaluating: 100%|██████████| 466/466 [00:16<00:00, 28.61it/s]
Epoch 1 complete. Accuracy: 87.188%

Starting Epoch 2...
Training: 100%|██████████| 3258/3258 [02:10<00:00, 24.96it/s]
Evaluating: 100%|██████████| 466/466 [00:16<00:00, 28.67it/s]
Epoch 2 complete. Accuracy: 90.317%

Starting Epoch 3...
Training: 100%|██████████| 3258/3258 [02:12<00:00, 24.64it/s]
Evaluating: 100%|██████████| 466/466 [00:15<00:00, 30.15it/s]
Epoch 3 complete. Accuracy: 92.318%

Starting Epoch 4...
Training: 100%|██████████| 3258/3258 [02:10<00:00, 24.88it/s]
Evaluating: 100%|██████████| 466/466 [00:15<00:00, 29.88it/s]
Epoch 4 complete. Accuracy: 93.795%

Starting Epoch 5...
Training: 100%|██████████| 3258/3258 [02:10<00:00, 24.91it/s]
Evaluating: 100%|██████████| 466/466 [00:16<00:00, 28.65it/s]
Epoch 5 complete. Accuracy: 93.809%

50 tokens, 5 epch, 16 batch size

TRAINING START. Using Device: cuda
Starting Epoch 1...
Training: 100%|██████████| 3258/3258 [02:14<00:00, 24.19it/s]
Evaluating: 100%|██████████| 466/466 [00:15<00:00, 30.71it/s]
Epoch 1 complete. Accuracy: 90.787%

Starting Epoch 2...
Training: 100%|██████████| 3258/3258 [02:15<00:00, 24.08it/s]
Evaluating: 100%|██████████| 466/466 [00:16<00:00, 28.88it/s]
Epoch 2 complete. Accuracy: 92.600%

Starting Epoch 3...
Training: 100%|██████████| 3258/3258 [02:09<00:00, 25.07it/s]
Evaluating: 100%|██████████| 466/466 [00:15<00:00, 30.87it/s]
Epoch 3 complete. Accuracy: 94.306%

Starting Epoch 4...
Training: 100%|██████████| 3258/3258 [02:14<00:00, 24.29it/s]
Evaluating: 100%|██████████| 466/466 [00:15<00:00, 29.82it/s]
Epoch 4 complete. Accuracy: 95.111%

Starting Epoch 5...
Training: 100%|██████████| 3258/3258 [02:14<00:00, 24.18it/s]
Evaluating: 100%|██████████| 466/466 [00:15<00:00, 30.71it/s]
Epoch 5 complete. Accuracy: 95.058%

5EPCH, 50 tokens, 8 batch size

TRAINING START. Using Device: cuda
Starting Epoch 1...
Training: 100%|██████████| 6516/6516 [02:17<00:00, 47.46it/s]
Evaluating: 100%|██████████| 931/931 [00:16<00:00, 56.48it/s]
Epoch 1 complete. Accuracy: 87.779%

Starting Epoch 2...
Training: 100%|██████████| 6516/6516 [02:17<00:00, 47.30it/s]
Evaluating: 100%|██████████| 931/931 [00:17<00:00, 52.40it/s]
Epoch 2 complete. Accuracy: 92.103%

Starting Epoch 3...
Training: 100%|██████████| 6516/6516 [02:18<00:00, 46.99it/s]
Evaluating: 100%|██████████| 931/931 [00:16<00:00, 54.80it/s]
Epoch 3 complete. Accuracy: 94.037%

Starting Epoch 4...
Training: 100%|██████████| 6516/6516 [02:18<00:00, 46.92it/s]
Evaluating: 100%|██████████| 931/931 [00:16<00:00, 55.12it/s]
Epoch 4 complete. Accuracy: 94.064%

Starting Epoch 5...
Training: 100%|██████████| 6516/6516 [02:18<00:00, 47.05it/s]
Evaluating: 100%|██████████| 931/931 [00:17<00:00, 54.03it/s]
Epoch 5 complete. Accuracy: 95.353%
